<a href="https://colab.research.google.com/github/RuthBiney/Ampe-DB/blob/main/Ampe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##STEP 1 - Mount Drive
This mounts Google Drive to the Colab environment so the notebook can access and save files (videos, outputs, CSV, JSON) stored in Drive. All Drive files become available under /content/drive/MyDrive/.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##STEP 2


*   Imports the os module to handle file paths and check whether directories exist.

*   Imports the glob module to search for files using wildcard patterns.

*   Defines the root input folder that contains all video clips and their subfolders.

*   Verifies that the input folder exists; if not, execution stops with a clear error message.

*   Specifies common video file formats (.mp4, .avi, .mov, .mkv) to ensure all relevant videos are included.

*   Uses recursive search (**/) to locate video files inside all nested subdirectories.

*   Collects the full file paths of all discovered videos into a single list.

*   Prints the total number of videos found to confirm successful dataset discovery.

*   Displays a preview of the first few video paths for quick visual validation of the folder structure.

In [ ]:
import os, glob

INPUT_FOLDER = "/content/drive/MyDrive/Ampe_Dataset/Videos/Clips "

if not os.path.exists(INPUT_FOLDER):
    raise FileNotFoundError(f"Folder not found: {INPUT_FOLDER}")

# recursive search pattern
patterns = ["**/*.mp4", "**/*.avi", "**/*.mov", "**/*.mkv"]
files = []

for p in patterns:
    files.extend(glob.glob(os.path.join(INPUT_FOLDER, p), recursive=True))

print(f"Total video files found (including subfolders): {len(files)}\n")

# preview first 30 items
for i, f in enumerate(files[:30], 1):
    print(f"{i:02d}. {f}")

Total video files found (including subfolders): 637

01. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C004.mp4
02. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C005.mp4
03. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C006.mp4
04. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C007.mp4
05. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C008.mp4
06. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C010.mp4
07. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C009.mp4
08. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C011.mp4
09. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C012.mp4
10. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C013.mp4
11. /content/drive/MyDrive/Ampe_Dataset/Videos/Clips /right_left/right_left_C014.mp4
12. /content

##STEP 3 - Remove Existing MediaPipe Installations

*   Uninstalls any previously installed versions of mediapipe and mediapipe-silicon.

*   Prevents version conflicts and binary incompatibility issues (common in Colab).

*   Ensures a clean environment before reinstalling the correct MediaPipe version.

*   Helps avoid errors related to mismatched NumPy or Python builds.

In [ ]:
!pip uninstall -y mediapipe mediapipe-silicon


*   Installs a specific, known-stable version of MediaPipe (0.10.21) compatible with Colab.

*   Uses the Google Coral package index to obtain pre-built MediaPipe wheels. Includes PyPI as a fallback index to resolve required dependencies.

In [ ]:
!pip install mediapipe==0.10.21 --index-url https://google-coral.github.io/py-repo/ --extra-index-url https://pypi.org/simple

Looking in indexes: https://google-coral.github.io/py-repo/, https://pypi.org/simple
Looking in indexes: https://google-coral.github.io/py-repo/, https://pypi.org/simple


###Install YOLO and OpenCV Dependencies

*   Installs a specific stable version of Ultralytics YOLO (8.2.103) for person detection.

*   Ensures consistent YOLO model behavior across different environments.

*   Installs OpenCV for video reading, frame processing, and skeleton rendering.

*   Completes the dependency setup required for detection, pose estimation, and visualization.


In [ ]:
!pip install ultralytics==8.2.103 --quiet
!pip install opencv-python --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.1/875.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 9.3 MB/s eta 0:00:00


##STEP 4 - Import Libraries and Load Models

*   Imports MediaPipe for human pose (skeleton) estimation.

*   Imports Ultralytics YOLO for detecting people in video frames.

*   Imports OpenCV for video handling and image processing.

*   Imports NumPy for numerical operations and coordinate calculations.

*   Imports os for file and path management.

*   Loads a pretrained YOLOv8n model (yolov8n.pt) optimized for fast person detection and restricts YOLO usage to detecting humans in later steps of the pipeline.

*   Initializes the MediaPipe Pose model for full-body landmark detection which Enables landmark smoothing to reduce jitter across frames.

*   Configures the model for continuous video processing rather than static images.

*   Prints a confirmation message indicating that all models are successfully loaded and ready for use.



In [ ]:
import mediapipe as mp
from ultralytics import YOLO
import cv2
import numpy as np
import os

# Load YOLO person detector
yolo_model = YOLO("yolov8n.pt")

# MediaPipe pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False,
                    model_complexity=1,
                    smooth_landmarks=True,
                    enable_segmentation=False,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)

print("Models loaded!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 6.25M/6.25M [00:00<00:00, 102MB/s]


Models loaded!


##STEP 5 - Process All Videos and Generate Skeleton Outputs

In this step, the code processes each input video and generates three different outputs:

1.   A skeleton-only video

2.   A CSV file containing numerical pose keypoints

3.   A JSON file containing structured frame-by-frame annotations.

To keep the dataset well-organized, all outputs are saved inside a main folder called Skeleton_Outputs, with separate subfolders for each data type.

####📂 Folder Structure Created


*   Skeleton_Outputs/videos/ → skeleton-only videos

*   Skeleton_Outputs/csv/ → pose keypoints in tabular format

*   Skeleton_Outputs/json/ → pose keypoints in structured annotation format

This structure makes it easier to:

*   Access only the data type you need

*   Train models using CSV files

*   Visualize results using skeleton videos

*   Load annotations cleanly for analysis or labeling

####⚙️ What the Code Does

*   Recursively scans the input dataset folder and finds all video files.

*   Processes each video frame-by-frame.

*   Uses YOLO to detect up to two players per frame.

*   Uses MediaPipe Pose to extract body keypoints from each detected player.

*   Draws clean skeletons on a blank background (no original video).

*   Saves:

    *   The skeleton animation as a video file

    *   The pose coordinates as a CSV file

    *   The full frame-level annotations as a JSON file

*   Automatically places each output into its corresponding subfolder.



In [ ]:
import os
import cv2
import numpy as np
import json
import pandas as pd
from tqdm import tqdm

# -------------------------
# Config
# -------------------------
INPUT_ROOT = "/content/drive/MyDrive/Ampe_Dataset/Videos/Clips "
OUTPUT_ROOT = "/content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs"

# subfolders
VIDEO_OUT = os.path.join(OUTPUT_ROOT, "videos")
CSV_OUT   = os.path.join(OUTPUT_ROOT, "csv")
JSON_OUT  = os.path.join(OUTPUT_ROOT, "json")

os.makedirs(VIDEO_OUT, exist_ok=True)
os.makedirs(CSV_OUT, exist_ok=True)
os.makedirs(JSON_OUT, exist_ok=True)

OUTPUT_WIDTH = 1280
OUTPUT_HEIGHT = 720
OUTPUT_FPS = 25

PLAYER_COLORS = [(0, 0, 255), (255, 0, 0)]  # Player1=Red, Player2=Blue
MIN_VISIBILITY = 0.3

POSE_CONNECTIONS = mp_pose.POSE_CONNECTIONS

# -------------------------
# helpers
# -------------------------
def find_all_videos(root):
    exts = {".mp4", ".mov", ".avi", ".mkv"}
    vids = []
    for r, _, files in os.walk(root):
        for f in files:
            if os.path.splitext(f)[1].lower() in exts:
                vids.append(os.path.join(r, f))
    return sorted(vids)

def sort_boxes_left_to_right(boxes):
    return sorted(boxes, key=lambda b: (b[0] + b[2]) / 2)

# -------------------------
# process one video
# -------------------------
def process_video(video_path):
    base = os.path.splitext(os.path.basename(video_path))[0]
    print(f"\n--- Processing: {base} ---")

    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print("  ERROR: cannot open", video_path)
            return

        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        in_fps = cap.get(cv2.CAP_PROP_FPS)
        fps = in_fps if in_fps and in_fps > 1 else OUTPUT_FPS

        # video output
        out_video_path = os.path.join(VIDEO_OUT, f"skeleton_{base}.mp4")
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(out_video_path, fourcc, fps, (OUTPUT_WIDTH, OUTPUT_HEIGHT))

        json_data = {"video": base, "source": video_path, "frames": []}
        csv_rows = []

        frame_idx = 0
        pbar = tqdm(total=frame_count, desc=f"Frames {base}", leave=False)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_idx += 1
            h, w = frame.shape[:2]

            results = yolo_model(frame, classes=[0], verbose=False)
            boxes = []

            if results and results[0].boxes is not None:
                for bx in results[0].boxes:
                    xyxy = bx.xyxy[0].cpu().numpy()
                    x1, y1, x2, y2 = map(int, xyxy[:4])
                    score = float(bx.conf[0])
                    boxes.append((x1, y1, x2, y2, score))

            boxes = sort_boxes_left_to_right(boxes)[:2]

            canvas = np.ones((OUTPUT_HEIGHT, OUTPUT_WIDTH, 3), dtype=np.uint8) * 255
            frame_entry = {"frame_index": frame_idx, "players": []}

            for pidx, (x1, y1, x2, y2, _) in enumerate(boxes):
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w-1, x2), min(h-1, y2)
                if x2 <= x1 or y2 <= y1:
                    continue

                crop = frame[y1:y2, x1:x2]
                rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                res = pose.process(rgb)

                lm_list = []
                if res.pose_landmarks:
                    for li, lm in enumerate(res.pose_landmarks.landmark):
                        abs_x = lm.x * (x2 - x1) + x1
                        abs_y = lm.y * (y2 - y1) + y1
                        out_x = int(abs_x / w * OUTPUT_WIDTH)
                        out_y = int(abs_y / h * OUTPUT_HEIGHT)

                        lm_list.append({
                            "index": li,
                            "x_pixel_out": out_x,
                            "y_pixel_out": out_y,
                            "visibility": lm.visibility
                        })

                        csv_rows.append({
                            "video": base,
                            "frame": frame_idx,
                            "player_id": pidx + 1,
                            "landmark_index": li,
                            "x_pixel_out": out_x,
                            "y_pixel_out": out_y,
                            "visibility": lm.visibility
                        })

                frame_entry["players"].append({
                    "player_id": pidx + 1,
                    "landmarks": lm_list
                })

                color = PLAYER_COLORS[pidx]
                for lm in lm_list:
                    if lm["visibility"] >= MIN_VISIBILITY:
                        cv2.circle(canvas, (lm["x_pixel_out"], lm["y_pixel_out"]), 4, color, -1)

                for a, b in POSE_CONNECTIONS:
                    if a < len(lm_list) and b < len(lm_list):
                        la, lb = lm_list[a], lm_list[b]
                        if la["visibility"] >= MIN_VISIBILITY and lb["visibility"] >= MIN_VISIBILITY:
                            cv2.line(canvas,
                                     (la["x_pixel_out"], la["y_pixel_out"]),
                                     (lb["x_pixel_out"], lb["y_pixel_out"]),
                                     color, 3)

            json_data["frames"].append(frame_entry)
            writer.write(canvas)
            pbar.update(1)

        cap.release()
        writer.release()
        pbar.close()

        # save annotations
        csv_path = os.path.join(CSV_OUT, f"keypoints_{base}.csv")
        json_path = os.path.join(JSON_OUT, f"keypoints_{base}.json")

        pd.DataFrame(csv_rows).to_csv(csv_path, index=False)
        with open(json_path, "w") as f:
            json.dump(json_data, f, indent=2)

        print("Saved video:", out_video_path)
        print("Saved csv  :", csv_path)
        print("Saved json :", json_path)

    except Exception as e:
        print("ERROR:", e)

# -------------------------
# run all videos
# -------------------------
videos = find_all_videos(INPUT_ROOT)
print("Videos discovered:", len(videos))

for v in videos:
    process_video(v)

print("\nALL DONE.")


Videos discovered: 637

--- Processing: left_left_C001 ---


Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_left_left_C001.mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_left_left_C001.csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_left_left_C001.json

--- Processing: left_left_C002 ---


Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_left_left_C002.mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_left_left_C002.csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_left_left_C002.json

--- Processing: left_left_C003 ---


Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_left_left_C003.mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_left_left_C003.csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_left_left_C003.json

--- Processing: left_left_C004 ---


Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_left_left_C004.mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_left_left_C004.csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_left_left_C004.json

--- Processing: left_left_C005 ---


...

Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_right_right_C185.mp4 .mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_right_right_C185.mp4 .csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_right_right_C185.mp4 .json

--- Processing: right_right_C186.mp4  ---


Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_right_right_C186.mp4 .mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_right_right_C186.mp4 .csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_right_right_C186.mp4 .json

--- Processing: right_right_C187.mp4  ---


Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_right_right_C187.mp4 .mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_right_right_C187.mp4 .csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_right_right_C187.mp4 .json

--- Processing: right_right_C188.mp4  ---


Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_right_right_C188.mp4 .mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_right_right_C188.mp4 .csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_right_right_C188.mp4 .json

--- Processing: right_right_C189.mp4  ---


Saved video: /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/videos/skeleton_right_right_C189.mp4 .mp4
Saved csv  : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv/keypoints_right_right_C189.mp4 .csv
Saved json : /content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/json/keypoints_right_right_C189.mp4 .json

ALL DONE.


###Step 6 — Normalize Pose Keypoints (Dataset Preparation)

####1️⃣ Load All Keypoint CSV Files

1. The script scans the skeleton output folder and loads every CSV file that contains pose keypoint data.

2. Each CSV file corresponds to a single video clip and contains frame-level skeletal coordinates for both players.

2️⃣ Process Frame-by-Frame and Player-by-Player

For each video:

*   Data is grouped by frame number
*   Data is further grouped by player ID

This ensures that each pose instance represents one player in one frame.

Because Ampe is a paired interaction dataset, both players are processed independently within each frame.

####3️⃣ Identify Reference Joints

Four MediaPipe landmarks are required:

*   Left hip
*   Right hip
*   Left shoulder
*   Right shoulder

These joints are used to define the body center and body scale.

####4️⃣ Compute Normalization Reference

The following reference points are computed:

*   Pelvis center: average of left and right hips
*   Shoulder center: average of left and right shoulders
*   Torso length: distance between pelvis center and shoulder center

This torso length is used as the scaling factor for normalization.

####5️⃣ Normalize All Joints

For every landmark:

*   Subtract the pelvis center coordinates
*   Divide by torso length

This produces:

*   x_norm
*   y_norm

The pose now depends only on movement dynamics and not on body size, camera distance, or video resolution.

####6️⃣ Save the Normalized Files Per Video

Instead of combining all normalized rows into one large master CSV file, each processed video is now saved as its own normalized CSV file.

Example output files include:

*   left_left_C001_normalized.csv
*   left_right_C002_normalized.csv
*   right_left_C003_normalized.csv

All files are saved inside:

*   keypoints_normalized/

This improves:

*   file manageability
*   easier downloading
*   selective access
*   machine learning workflows
*   dataset scalability


In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# -------------------------
# PATHS
# -------------------------
INPUT_FOLDER = "/content/drive/MyDrive/Ampe_Dataset/Videos/Skeleton_Outputs/csv"
OUTPUT_FOLDER = "/content/drive/MyDrive/Ampe_Dataset/keypoints_normalized"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# MediaPipe landmark indices
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_HIP = 23
RIGHT_HIP = 24

csv_files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".csv")]

print("CSV files found:", len(csv_files))

for csv_file in tqdm(csv_files):

    video_id = csv_file.replace("keypoints_", "").replace(".csv", "")
    df = pd.read_csv(os.path.join(INPUT_FOLDER, csv_file))

    normalized_rows = []

    for (frame, player), group in df.groupby(["frame", "player_id"]):

        landmarks = {row.landmark_index: row for _, row in group.iterrows()}

        required = [LEFT_HIP, RIGHT_HIP, LEFT_SHOULDER, RIGHT_SHOULDER]

        if not all(j in landmarks for j in required):
            continue

        hip_x = (
            landmarks[LEFT_HIP].x_pixel_out +
            landmarks[RIGHT_HIP].x_pixel_out
        ) / 2

        hip_y = (
            landmarks[LEFT_HIP].y_pixel_out +
            landmarks[RIGHT_HIP].y_pixel_out
        ) / 2

        shoulder_x = (
            landmarks[LEFT_SHOULDER].x_pixel_out +
            landmarks[RIGHT_SHOULDER].x_pixel_out
        ) / 2

        shoulder_y = (
            landmarks[LEFT_SHOULDER].y_pixel_out +
            landmarks[RIGHT_SHOULDER].y_pixel_out
        ) / 2

        torso_length = np.sqrt(
            (shoulder_x - hip_x)**2 +
            (shoulder_y - hip_y)**2
        )

        if torso_length < 1e-6:
            continue

        for _, row in group.iterrows():

            x_norm = (row.x_pixel_out - hip_x) / torso_length
            y_norm = (row.y_pixel_out - hip_y) / torso_length

            normalized_rows.append({
                "frame": frame,
                "player_id": player,
                "landmark_index": row.landmark_index,
                "x_norm": x_norm,
                "y_norm": y_norm,
                "visibility": row.visibility
            })

    # SAVE EACH VIDEO SEPARATELY
    output_file = os.path.join(
        OUTPUT_FOLDER,
        f"{video_id}_normalized.csv"
    )

    pd.DataFrame(normalized_rows).to_csv(output_file, index=False)

print("Normalization complete")
print("Files saved in:", OUTPUT_FOLDER)

CSV files found: 637


100%|██████████| 637/637 [06:06<00:00,  1.74it/s]

Normalization complete
Files saved in: /content/drive/MyDrive/Ampe_Dataset/keypoints_normalized


###STEP 7 Generate Frame-Level Annotations (Label Creation)


####1️⃣ Load the Normalized Pose Dataset

The script loads the normalized pose keypoints CSV file from Google Drive.

This dataset contains:

*   Video ID
*   Frame number
*   Player ID
*   Landmark index
*   Normalized coordinates

The data is loaded into a pandas DataFrame for processing.


####2️⃣ Create Output Annotation Folder

1. The script creates the annotation output directory if it does not already exist.

2. This ensures the generated labels can be saved without any file path errors.


####3️⃣ Define the Key Landmarks for Leg Detection

Two MediaPipe pose landmarks are used:

*   Left ankle → 27
*   Right ankle → 28

These joints are important for identifying which leg each player has raised.

####4️⃣ Process Data Frame-by-Frame

The dataset is grouped by:

*   Video ID
*   Frame number

This allows the script to analyze one frame at a time for each video.

Each group contains all pose landmarks detected in that specific frame.

####5️⃣ Process Player-by-Player Within Each Frame

Inside every frame:

*   Data is grouped by player ID
*   Each player is processed separately

This ensures the movement of Player 1 and Player 2 can be analyzed independently.

####6️⃣ Extract Ankle Coordinates

For each player:

*   Left ankle y_norm is extracted
*   Right ankle y_norm is extracted

The vertical coordinate (y_norm) is used to determine which leg is raised.

####7️⃣ Detect the Raised Leg

The ankle positions are compared:

*   If left ankle is higher → label = L
*   Otherwise → label = R

Since image coordinates increase downward:

*   Smaller y value = higher leg position
*   List item

This step identifies whether the player lifted the left leg or right leg.

####8️⃣ Create Combined Frame Labels

Once both players are detected in the frame:

The labels are combined as:

*   Player 1 label + Player 2 label

Examples:

*   LL
*   LR
*   RL
*   RR

This creates the final Ampe movement label for that frame.

####9️⃣ Save All Frame Annotations

All generated labels are stored in a new CSV file containing:

*   Video ID
*   Frame number
*   Frame label

The final annotation dataset is saved for machine learning and action recognition tasks.

In [ ]:
import pandas as pd
import numpy as np
import os

INPUT_DATASET = "/content/drive/MyDrive/Ampe_Dataset/keypoints_normalized/Ampe_Pose_Dataset.csv"
OUTPUT_ANNOTATIONS = "/content/drive/MyDrive/Ampe_Dataset/annotation/Ampe_annotations.csv "

# Create the output directory if it does not exist
os.makedirs(os.path.dirname(OUTPUT_ANNOTATIONS), exist_ok=True)

df = pd.read_csv(INPUT_DATASET)

LEFT_ANKLE = 27
RIGHT_ANKLE = 28

annotations = []

for (video, frame), group in df.groupby(["video_id", "frame"]):

    players = {}

    for player, pgroup in group.groupby("player_id"):

        joints = {row.landmark_index: row for _, row in pgroup.iterrows()}

        if LEFT_ANKLE in joints and RIGHT_ANKLE in joints:

            left_y = joints[LEFT_ANKLE].y_norm
            right_y = joints[RIGHT_ANKLE].y_norm

            if left_y < right_y:
                players[player] = "L"
            else:
                players[player] = "R"

    if 1 in players and 2 in players:

        label = players[1] + players[2]

        annotations.append({
            "video_id": video,
            "frame": frame,
            "label": label
        })

annotations_df = pd.DataFrame(annotations)
annotations_df.to_csv(OUTPUT_ANNOTATIONS, index=False)

print("Annotation file created")
print("Saved:", OUTPUT_ANNOTATIONS)
print("Total labeled frames:", len(annotations_df))

Annotation file created
Saved: /content/drive/MyDrive/Ampe_Dataset/annotation/Ampe_annotations.csv 
Total labeled frames: 35004


##Step 8 — Generate Dataset Metadata

This code will automatically extract:

video ID

class label (left_left, left_right, etc.)

FPS

frame count

duration

resolution (width × height)

file size

and save everything to metadata/Ampe_metadata.csv.

In [ ]:
import os
import cv2
import pandas as pd

video_root =  "/content/drive/MyDrive/Ampe_Dataset/Videos/Clips "
metadata_dir = "/content/drive/MyDrive/Ampe_Dataset/metadata"

os.makedirs(metadata_dir, exist_ok=True)

metadata = []

video_extensions = (".mp4", ".MP4", ".mov", ".MOV", ".m4v")

for root, dirs, files in os.walk(video_root):

    for file in files:

        if not file.endswith(video_extensions):
            continue

        video_path = os.path.join(root, file)

        video_id = os.path.splitext(file)[0]
        label = os.path.basename(root)

        cap = cv2.VideoCapture(video_path)

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

        duration = frame_count / fps if fps > 0 else 0

        file_size = os.path.getsize(video_path) / (1024 * 1024)

        metadata.append([
            video_id,
            label,
            int(frame_count),
            fps,
            duration,
            int(width),
            int(height),
            round(file_size,2)
        ])

        cap.release()

metadata_df = pd.DataFrame(
    metadata,
    columns=[
        "video_id",
        "label",
        "frame_count",
        "fps",
        "duration_seconds",
        "width",
        "height",
        "file_size_MB"
    ]
)

metadata_path = os.path.join(metadata_dir, "Ampe_metadata.csv")

metadata_df.to_csv(metadata_path, index=False)

print("Metadata file saved:", metadata_path)
print("Total videos processed:", len(metadata_df))

Metadata file saved: /content/drive/MyDrive/Ampe_Dataset/metadata/Ampe_metadata.csv
Total videos processed: 637


##Step 9 — Generate Dataset Statistics

This code reads the metadata file created in Step 8 and computes useful statistics.


*   number of samples per class

*   average clip duration

*   dataset size

*   recording resolution

*   frame rate distribution



In [ ]:
import pandas as pd
import os

metadata_path = "/content/drive/MyDrive/Ampe_Dataset/metadata/Ampe_metadata.csv"
save_path = "/content/drive/MyDrive/Ampe_Dataset/metadata/dataset_statistics.csv"

df = pd.read_csv(metadata_path)

# Calculate statistics
total_clips = len(df)
avg_duration = df["duration_seconds"].mean()
total_duration = df["duration_seconds"].sum()
avg_fps = df["fps"].mean()
avg_width = df["width"].mean()
avg_height = df["height"].mean()
total_size = df["file_size_MB"].sum()

stats = {
    "Total Clips": total_clips,
    "Average Clip Duration (s)": round(avg_duration,2),
    "Total Dataset Duration (minutes)": round(total_duration/60,2),
    "Average FPS": round(avg_fps,2),
    "Average Width": int(avg_width),
    "Average Height": int(avg_height),
    "Total Dataset Size (GB)": round(total_size/1024,2)
}

stats_df = pd.DataFrame(list(stats.items()), columns=["Statistic","Value"])

stats_df.to_csv(save_path, index=False)

print("Dataset statistics saved to:")
print(save_path)

Dataset statistics saved to:
/content/drive/MyDrive/Ampe_Dataset/metadata/dataset_statistics.csv
